In [0]:
%run /Workspace/Users/chittoriarocks@gmail.com/Gold_files/framework/merge_helper


In [0]:
%run /Workspace/Users/chittoriarocks@gmail.com/Gold_files/framework/watermark_helper

In [0]:
from pyspark.sql import functions as F
table_name = "fact_claim"


# STEP 1 — GET WATERMARK

watermark = get_watermark(table_name)


# STEP 2 — READ INCREMENTAL CLAIMS

claims_df = (
    spark.table("silver.claims")
    .filter(
        F.col("updated_at") > F.lit(watermark)
    )
)


# STEP 3 — BUILD FACT DATASET

fact_claim_df = (
    claims_df
    .select(
        "claim_id",
        "customer_id",
        "policy_id",
        "claim_amount",
        "claim_status",
        "claim_date"
    )
    .withColumn(
        "claim_year",
        F.year("claim_date")
    )
    .withColumn(
        "gold_updated_ts",
        F.current_timestamp()
    )
)


# STEP 4 — MERGE

execute_merge(
    spark=spark,

    source_df=fact_claim_df,

    target_table="gold.fact_claim",

    merge_condition="""
        t.claim_id = s.claim_id
    """,

    update_map={
        "claim_amount":"s.claim_amount",
        "claim_status":"s.claim_status",
        "gold_updated_ts":"s.gold_updated_ts"
    },

    insert_map={
        "claim_id":"s.claim_id",
        "customer_id":"s.customer_id",
        "policy_id":"s.policy_id",
        "claim_amount":"s.claim_amount",
        "claim_status":"s.claim_status",
        "claim_date":"s.claim_date",
        "claim_year":"s.claim_year",
        "gold_updated_ts":"s.gold_updated_ts"
    }
)


# STEP 5 — UPDATE WATERMARK

update_watermark(table_name)